In [1]:
!pip install torch torchvision matplotlib

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.2 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import cv2
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [9]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor()])

train_data = datasets.ImageFolder(
    root="datasets/AlphabetASL/asl_alphabet_train/asl_alphabet_train",
    transform=transform,  # change this
)

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

print(train_data.classes)  # should show A-Z

['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


In [6]:
import torch
import torch.nn as nn
from torchvision import models

model = models.resnet18(pretrained=True)

# replace last layer for 26 classes
model.fc = nn.Linear(model.fc.in_features, 29)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

In [8]:
import torch

print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Current device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")

# More details if GPU is available
if torch.cuda.is_available():
    print(f"GPU name: {torch.cuda.get_device_name(0)}")
    print(f"GPU count: {torch.cuda.device_count()}")

CUDA available: False
Current device: cpu


In [10]:
from tqdm import tqdm
import torch.optim as optim

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(2):
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}")

    for images, labels in loop:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # update progress bar with loss
        loop.set_postfix(loss=loss.item())

    print(f"Epoch {epoch+1} done")

Epoch 1:   1%|          | 17/2719 [01:00<2:39:20,  3.54s/it, loss=0.143] 


KeyboardInterrupt: 